# Notebook 06d — NRAGLS-BERT (Frozen + Precompute)

Vì trọng số của BERT bị đóng băng toàn bộ, có thể cho BERT chạy qua toàn bộ tập tin tức một lần duy nhất ở đầu, lưu kết quả thành 1 tensor `(NUM_NEWS, 768)`.
Trong quá trình huấn luyện, User Encoder chỉ Lookup (tra cứu) các vector này thay vì chạy lại BERT

In [1]:
import sys, time, json, math, random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import Dataset, DataLoader

sys.path.insert(0, '.')
sys.path.append('/kaggle/input/datasets/neitng/utils-for-dl-major-assignment')

from utils import (
    seed_everything, TRAIN_DIR, DEV_DIR, WORK_DIR, MODEL_DIR, SEED,
    load_news, load_behaviors, parse_impressions,
    build_vocab, tokenize_to_ids,
    MINDTrainDataset, collate_train,
    compute_ranking_metrics, count_parameters
)

seed_everything(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'DEVICE: {DEVICE}')

PLM_NAME     = 'distilbert-base-uncased'
NEWS_DIM     = 128
# N_HEADS_NEWS (Not needed for BERT)
N_HEADS_USER = 8
MAX_TITLE    = 30
MAX_HIST     = 30
NEG_K        = 4
BATCH_SIZE   = 64
LR           = 5e-4
WEIGHT_DECAY = 1e-5
DROPOUT      = 0.1
EPOCHS       = 5
WARMUP_RATIO = 0.1

DEVICE: cuda


In [2]:
print("Loading data...")
news_train = load_news(TRAIN_DIR)
news_dev   = load_news(DEV_DIR)
news_all   = pd.concat([news_train, news_dev]).drop_duplicates('news_id').reset_index(drop=True)
beh_train  = load_behaviors(TRAIN_DIR)
beh_dev    = load_behaviors(DEV_DIR)

print(f"News: {len(news_all)}, Train: {len(beh_train)}, Dev: {len(beh_dev)}")

Loading data...
News: 65238, Train: 156965, Dev: 73152


In [3]:
print("Loading HuggingFace Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(PLM_NAME)

all_nids = news_all['news_id'].tolist()
nid2idx = {n: i+1 for i, n in enumerate(all_nids)}
NUM_NEWS = len(nid2idx)

print("Tokenizing titles and abstracts...")
texts = []
for _, row in news_all.iterrows():
    text = str(row['title'] or '') + ' ' + str(row['abstract'] or '')
    texts.append(text)

encoded = tokenizer(texts, padding='max_length', truncation=True, max_length=MAX_TITLE, return_tensors='pt')

# Create tensors + 1 dummy for padding idx 0
news_input_ids = torch.zeros(NUM_NEWS + 1, MAX_TITLE, dtype=torch.long)
news_attention_mask = torch.zeros(NUM_NEWS + 1, MAX_TITLE, dtype=torch.long)

news_input_ids[1:] = encoded['input_ids']
news_attention_mask[1:] = encoded['attention_mask']

print(f"Tokenized {NUM_NEWS} news articles into tensors of shape {news_input_ids.shape}")

Loading HuggingFace Tokenizer...


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizing titles and abstracts...
Tokenized 65238 news articles into tensors of shape torch.Size([65239, 30])


In [4]:
print("Precomputing DistilBERT embeddings for all news (chạy 1 lần duy nhất)...")
plm = AutoModel.from_pretrained(PLM_NAME).to(DEVICE)
plm.eval()

precomputed_embs = torch.zeros(NUM_NEWS + 1, plm.config.hidden_size)
batch_sz = 256

with torch.no_grad():
    for i in range(1, NUM_NEWS + 1, batch_sz):
        end = min(i + batch_sz, NUM_NEWS + 1)
        b_ids = news_input_ids[i:end].to(DEVICE)
        b_mask = news_attention_mask[i:end].to(DEVICE)
        outputs = plm(input_ids=b_ids, attention_mask=b_mask)
        # Dùng CLS token
        cls_output = outputs.last_hidden_state[:, 0, :]
        precomputed_embs[i:end] = cls_output.cpu()
        if i % 10000 == 1:
            print(f"Processed {i-1}/{NUM_NEWS} articles...")

print(f"Finished. Precomputed tensor shape: {precomputed_embs.shape}")
del plm # Xóa model BERT khỏi VRAM để nhường chỗ cho model training
torch.cuda.empty_cache()


Precomputing DistilBERT embeddings for all news (chạy 1 lần duy nhất)...


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Processed 0/65238 articles...
Finished. Precomputed tensor shape: torch.Size([65239, 768])


In [5]:
class NewsEncoder(nn.Module):
    """Precomputed BERT-based News Encoder."""
    def __init__(self, precomputed_embs, news_dim, dropout):
        super().__init__()
        # Nạp tensor đã tính sẵn vào nn.Embedding và khóa lại
        self.emb = nn.Embedding.from_pretrained(precomputed_embs, freeze=True)
        
        plm_hidden_size = precomputed_embs.shape[1]
        self.proj = nn.Linear(plm_hidden_size, news_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, nid_indices):
        x = self.emb(nid_indices) # Lookup vector đã tính sẵn
        x = self.dropout(x)
        return self.proj(x)

class AdditiveAttention(nn.Module):
    def __init__(self, dim, hidden=64):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(dim, hidden), nn.Tanh(), nn.Linear(hidden, 1))

    def forward(self, x, mask=None):
        w = self.proj(x).squeeze(-1)
        if mask is not None:
            w = w.masked_fill(mask, -1e4)
        w = torch.softmax(w, dim=-1)
        w = torch.nan_to_num(w, nan=0.0)
        return (x * w.unsqueeze(-1)).sum(dim=1)


In [6]:
class RMSNorm(nn.Module):
    """Root Mean Square Layer Normalization (lighter than LayerNorm)."""
    def __init__(self, dim, eps=1e-8):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        rms = torch.sqrt(torch.mean(x ** 2, dim=-1, keepdim=True) + self.eps)
        return self.scale * x / rms


class GatedLinearAttention(nn.Module):
    """
    Gated Linear Attention with Recency Decay: O(L · d²) complexity.

    Instead of: softmax(QK⊤/√d) V  →  O(L² · d)
    Computes:   φ(Q) · (φ(K)⊤ · V)  →  O(L · d²)

    Improvements over original NRAGLS:
    - Content-based gating: g = σ(Wx+b) filters noisy clicks
    - Recency decay (OUR CONTRIBUTION): g_final = g · α^(L-i)
      α is learnable, automatically downweights older history items.
      This reflects real-world temporal user preference patterns.
    """
    def __init__(self, dim, n_heads, dropout=0.1):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = dim // n_heads
        assert dim % n_heads == 0

        self.W_q = nn.Linear(dim, dim)
        self.W_k = nn.Linear(dim, dim)
        self.W_v = nn.Linear(dim, dim)
        self.W_o = nn.Linear(dim, dim)

        # Content-based gating (from NRAGLS paper)
        self.gate = nn.Sequential(
            nn.Linear(dim, dim),
            nn.Sigmoid()
        )

        # ── OUR CONTRIBUTION: Recency Decay ──
        # Learnable decay factor α ∈ (0, 1), initialized at ~0.95
        # g_final(i) = g(i) · α^(L-i)
        # Recent items (small L-i) → decay ≈ 1.0 (keep)
        # Old items (large L-i) → decay → 0.0 (suppress)
        self._log_alpha = nn.Parameter(torch.tensor(math.log(0.95)))

        self.dropout = nn.Dropout(dropout)

    @property
    def alpha(self):
        """Decay factor constrained to (0, 1) via sigmoid."""
        return torch.sigmoid(self._log_alpha)

    def _feature_map(self, x):
        """Kernel feature map φ(x) = elu(x) + 1 (ensures non-negativity)."""
        return F.elu(x) + 1.0

    def _recency_decay(self, L, device):
        """Compute positional decay weights: α^(L-1-i) for i=0..L-1."""
        positions = torch.arange(L, device=device, dtype=torch.float32)
        # positions[0]=0 (oldest), positions[L-1]=L-1 (newest)
        # decay = α^(L-1-i): newest=α^0=1, oldest=α^(L-1)→small
        decay = self.alpha ** (L - 1 - positions)  # (L,)
        return decay.view(1, L, 1, 1)  # broadcast: (1, L, 1, 1)

    def forward(self, x, mask=None):
        B, L, D = x.shape
        H, d = self.n_heads, self.head_dim

        # Project Q, K, V
        Q = self._feature_map(self.W_q(x)).view(B, L, H, d)
        K = self._feature_map(self.W_k(x)).view(B, L, H, d)
        V = self.W_v(x).view(B, L, H, d)

        # Content-based gate (from paper)
        g = self.gate(x).view(B, L, H, d)  # (B, L, H, d)

        # ── OUR CONTRIBUTION: Recency-modulated gating ──
        # Multiply content gate by positional decay
        decay = self._recency_decay(L, x.device)  # (1, L, 1, 1)
        g = g * decay  # recent items keep high gate, old items suppressed

        K = K * g
        V = V * g

        # Mask padding positions
        if mask is not None:
            pad_mask = (~mask).float().view(B, L, 1, 1)  # 1=valid, 0=pad
            K = K * pad_mask
            V = V * pad_mask

        # Linear Attention: Q @ (K⊤ @ V) → O(L · d²)
        KV = torch.einsum('blhd,blhe->bhde', K, V)  # (B, H, d, d)
        out = torch.einsum('blhd,bhde->blhe', Q, KV)  # (B, L, H, d)

        # Normalize by sum of keys — clamp prevents NaN in backward
        K_sum = K.sum(dim=1)  # (B, H, d)
        denom = torch.einsum('blhd,bhd->blh', Q, K_sum).unsqueeze(-1)
        denom = denom.clamp(min=0.1)  # strong clamp — avoids 1/~0 gradient explosion
        out = out / denom

        out = torch.nan_to_num(out.reshape(B, L, D), nan=0.0)
        return self.dropout(self.W_o(out))


class SGLU(nn.Module):
    """Simplified Gated Linear Unit feed-forward."""
    def __init__(self, dim, expansion=2, dropout=0.1):  # expansion=2 for smaller model
        super().__init__()
        hidden = dim * expansion
        self.W1 = nn.Linear(dim, hidden)
        self.W2 = nn.Linear(dim, hidden)
        self.W_out = nn.Linear(hidden, dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.W_out(F.silu(self.W1(x)) * self.W2(x)))


class GLAUserEncoderLayer(nn.Module):
    """Single Gated Linear Attention layer + SGLU + RMSNorm."""
    def __init__(self, dim, n_heads, dropout=0.1):
        super().__init__()
        self.norm1 = RMSNorm(dim)
        self.gla = GatedLinearAttention(dim, n_heads, dropout)
        self.norm2 = RMSNorm(dim)
        self.ffn = SGLU(dim, expansion=4, dropout=dropout)

    def forward(self, x, mask=None):
        x = x + self.gla(self.norm1(x), mask)
        x = x + self.ffn(self.norm2(x))
        return x


class GLAUserEncoder(nn.Module):
    """User Encoder with Gated Linear Attention (replaces NRMS Self-Attention)."""
    def __init__(self, news_dim, n_heads, n_layers=2, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            GLAUserEncoderLayer(news_dim, n_heads, dropout)
            for _ in range(n_layers)
        ])
        self.attn_pool = AdditiveAttention(news_dim)

    def forward(self, news_vecs, mask=None):
        x = news_vecs
        for layer in self.layers:
            x = layer(x, mask)
        return self.attn_pool(x, mask)

In [7]:
class NRAGLS_BERT(nn.Module):
    def __init__(self, precomputed_embs, news_dim, n_heads_user, dropout, n_user_layers=2):
        super().__init__()
        self.news_encoder = NewsEncoder(precomputed_embs, news_dim, dropout)
        self.user_encoder = GLAUserEncoder(news_dim, n_heads_user,
                                           n_layers=n_user_layers,
                                           dropout=dropout)
        self.news_dim = news_dim

    def encode_news(self, nid_indices):
        return self.news_encoder(nid_indices)

    def encode_user(self, hist_indices):
        mask = (hist_indices == 0)
        news_vecs = self.encode_news(hist_indices)
        return self.user_encoder(news_vecs, mask)

    def forward(self, hist, pos, neg):
        u = self.encode_user(hist)
        p_vec = self.encode_news(pos)
        n_vec = self.encode_news(neg)
        pos_score = (u * p_vec).sum(-1, keepdim=True)
        neg_score = (u.unsqueeze(1) * n_vec).sum(-1)
        return torch.cat([pos_score, neg_score], dim=1)

    def score_candidates(self, hist, cand_indices):
        u = self.encode_user(hist)
        c_vec = self.encode_news(cand_indices)
        return (u * c_vec).sum(-1)


In [8]:
model = NRAGLS_BERT(
    precomputed_embs=precomputed_embs,
    news_dim=NEWS_DIM,
    n_heads_user=N_HEADS_USER,
    dropout=DROPOUT,
    n_user_layers=2,
).to(DEVICE)

n_params = count_parameters(model)
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"NRAGLS-BERT (Precompute) total parameters: {n_params:,} ({n_params/1e6:.2f}M)")
print(f"NRAGLS-BERT (Precompute) trainable params: {trainable_params:,} ({trainable_params/1e6:.2f}M)")


NRAGLS-BERT (Precompute) total parameters: 667,907 (0.67M)
NRAGLS-BERT (Precompute) trainable params: 667,907 (0.67M)


In [9]:
print("Building training dataset...")
train_ds = MINDTrainDataset(beh_train, nid2idx, max_hist=MAX_HIST,
                            neg_k=NEG_K, max_rows=150000)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                      collate_fn=lambda b: collate_train(b, MAX_HIST),
                      num_workers=2, pin_memory=True)
print(f"Training samples: {len(train_ds)}")

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
loss_fn = nn.CrossEntropyLoss()

# Linear warmup + cosine decay scheduler
total_steps = len(train_dl) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
def lr_lambda(step):
    if step < warmup_steps:
        return step / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1.0 + math.cos(math.pi * progress))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

epoch_times = []
for epoch in range(1, EPOCHS + 1):
    model.train()
    losses = []
    t0 = time.time()
    for H, P, N in train_dl:
        H, P, N = H.to(DEVICE), P.to(DEVICE), N.to(DEVICE)
        optimizer.zero_grad()
        logits = model(H, P, N)
        target = torch.zeros(logits.size(0), dtype=torch.long, device=DEVICE)
        loss = loss_fn(logits, target)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        losses.append(loss.item())

    dt = time.time() - t0
    epoch_times.append(dt)
    print(f"Epoch {epoch}/{EPOCHS} | loss={np.mean(losses):.5f} | time={dt:.1f}s")

# Print learned decay factor
for layer in model.user_encoder.layers:
    alpha_val = layer.gla.alpha.item()
    print(f"  Learned decay α = {alpha_val:.4f} (half-life ≈ {-1/math.log(alpha_val+1e-8):.1f} positions)")

Building training dataset...
Training samples: 150000
Epoch 1/5 | loss=1.48304 | time=35.9s
Epoch 2/5 | loss=1.43337 | time=35.0s
Epoch 3/5 | loss=1.39731 | time=35.2s
Epoch 4/5 | loss=1.35325 | time=35.2s
Epoch 5/5 | loss=1.31306 | time=35.2s
  Learned decay α = 0.6452 (half-life ≈ 2.3 positions)
  Learned decay α = 0.6760 (half-life ≈ 2.6 positions)


In [10]:
print("\nEvaluating on dev set...")
from utils import evaluate_model
metrics = evaluate_model(model, beh_dev, nid2idx, None,
                         max_hist=MAX_HIST, device=DEVICE)
print("=" * 50)
print("NRAGLS+ Results (with Recency Decay):")
for k, v in metrics.items():
    print(f"  {k}: {v:.6f}")
print("=" * 50)


Evaluating on dev set...
NRAGLS+ Results (with Recency Decay):
  AUC: 0.641438
  MRR: 0.347943
  nDCG@5: 0.330524
  nDCG@10: 0.393762


In [11]:
results = {
    'model': 'NRAGLS-BERT',
    'type': 'proposed',
    'complexity': 'O(L * d^2)',
    'parameters': n_params,
    'metrics': metrics,
    'epoch_times_sec': epoch_times,
    'avg_epoch_time_sec': float(np.mean(epoch_times)),
    'hyperparams': {
        'plm_name': PLM_NAME, 'news_dim': NEWS_DIM,
         'n_heads_user': N_HEADS_USER,
        'max_title': MAX_TITLE, 'max_hist': MAX_HIST,
        'neg_k': NEG_K, 'batch_size': BATCH_SIZE,
        'lr': LR, 'epochs': EPOCHS, 'dropout': DROPOUT,
        'n_user_layers': 2,
    }
}

torch.save(model.state_dict(), MODEL_DIR / 'nragls_bert_precompute.pt')
with open(WORK_DIR / 'nragls_bert_precompute_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f"\nModel saved to {MODEL_DIR / 'nragls_bert_precompute.pt'}")
print(f"Results saved to {WORK_DIR / 'nragls_bert_precompute_results.json'}")


Model saved to /kaggle/working/dl_results/models/nragls_bert_precompute.pt
Results saved to /kaggle/working/dl_results/nragls_bert_precompute_results.json
